In [ ]:
# Fix Windows asyncio/zmq warning and ensure project root is on sys.path
import asyncio
from asyncio import WindowsSelectorEventLoopPolicy
asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())
import sys
from pathlib import Path
# ensure project root is on sys.path so `import src` works
cwd = Path.cwd()
if cwd.name == 'data' and cwd.parent.exists():
    project_root = cwd.parent
else:
    project_root = cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import ta  # technical analysis library
import os
from src.features import generate_features


In [ ]:
# Load a few sample stocks (use Path for robust path handling)
DATA_DIR = Path(project_root) / 'data'
tickers = [p.stem for p in DATA_DIR.iterdir() if p.suffix == '.csv'][:5]
print(f"Working with {len(tickers)} tickers: {tickers}")

data = {t: pd.read_csv(DATA_DIR / f"{t}.csv", index_col=0, parse_dates=True) for t in tickers}


In [ ]:
processed_data = {}
for t, df in data.items():
    processed_data[t] = generate_features(df)
    print(f"{t}: {processed_data[t].shape}")


In [ ]:
# Concatenate all stocks together for training
combined = pd.concat(processed_data.values(), keys=processed_data.keys(), names=["Ticker", "Date"])
combined.dropna(inplace=True)
combined.to_csv("data/processed_features.csv")

print("✅ Saved combined feature file at data/processed_features.csv")
combined.head()


In [ ]:
import matplotlib.pyplot as plt

sample = combined.xs(tickers[0], level="Ticker")
sample[["Close", "rsi_14", "macd", "sma_10", "volatility_20"]].plot(subplots=True, figsize=(10,8))
plt.tight_layout()
plt.show()
